# BaseAttentive Standalone Applications

This notebook demonstrates practical applications of BaseAttentive as a standalone time series forecasting model for various real-world scenarios.

## Table of Contents
1. Air Quality Forecasting
2. Energy Demand Forecasting  
3. Weather Prediction
4. Traffic Flow Prediction

---

## Setup and Imports

In [ ]:
import warnings

import numpy as np

warnings.filterwarnings("ignore")

# BaseAttentive imports
from base_attentive import BaseAttentive
from base_attentive.backend import set_backend

# Set PyTorch backend for GPU support
set_backend("tensorflow")  # Or 'torch' if PyTorch installed

print("Setup complete!")

## 1. Air Quality Forecasting

### Use Case
Predict PM2.5 air pollution levels based on:
- **Static features**: Location (latitude, longitude), altitude, urban/rural indicator
- **Dynamic past features**: Historical PM2.5, NO₂, O₃, temperature, humidity
- **Known future features**: Forecasted wind speed, temperature

### Application
- Public health alerts
- Air quality advisories
- Industrial emission controls

In [ ]:
# Create synthetic air quality data
np.random.seed(42)
n_samples = 365 * 24  # 1 year of hourly data
batch_size = 32

# Static features (constant per location): [latitude, longitude, altitude, urban_indicator]
static_features = np.random.randn(batch_size, 4)
static_features[:, 2] = np.abs(static_features[:, 2]) * 1000  # Altitude 0-1000m
static_features[:, 3] = (static_features[:, 3] > 0).astype(float)  # Urban: 0 or 1

# Dynamic past features: [PM2.5, NO2, O3, Temperature, Humidity]
lookback_window = 168  # 7 days of hourly data
n_dynamic_past = 5
dynamic_past = np.random.randn(batch_size, lookback_window, n_dynamic_past)
# Normalize to realistic ranges
dynamic_past[:, :, 0] = np.abs(dynamic_past[:, :, 0]) * 50  # PM2.5: 0-100 µg/m³
dynamic_past[:, :, 1] = np.abs(dynamic_past[:, :, 1]) * 100  # NO2: 0-150 ppb
dynamic_past[:, :, 2] = np.abs(dynamic_past[:, :, 2]) * 50  # O3: 0-100 ppb
dynamic_past[:, :, 3] = 15 + dynamic_past[:, :, 3] * 10  # Temp: 5-25°C
dynamic_past[:, :, 4] = 50 + np.abs(dynamic_past[:, :, 4]) * 30  # Humidity: 20-80%

# Known future features: [forecasted_wind_speed, forecasted_temp]
forecast_horizon = 24  # 24-hour forecast
n_known_future = 2
known_future = np.random.randn(batch_size, forecast_horizon, n_known_future)
known_future[:, :, 0] = np.abs(known_future[:, :, 0]) * 5  # Wind speed: 0-10 m/s
known_future[:, :, 1] = 15 + known_future[:, :, 1] * 10  # Temp: 5-25°C

# Target: PM2.5 for next 24 hours
target = np.abs(np.random.randn(batch_size, forecast_horizon, 1)) * 50  # PM2.5

print("Air Quality Data Shapes:")
print(f"  Static features: {static_features.shape}")
print(f"  Dynamic past: {dynamic_past.shape}")
print(f"  Known future: {known_future.shape}")
print(f"  Target: {target.shape}")

In [ ]:
# Configure BaseAttentive for air quality forecasting
air_quality_model = BaseAttentive(
    static_dim=4,  # [lat, lon, alt, urban]
    dynamic_dim=5,  # [PM2.5, NO2, O3, Temp, Humidity]
    future_dim=2,  # [wind_speed, temp]
    forecast_horizon=24,  # 24-hour forecast
    lookback_window=168,  # 7 days historical
    mode="hybrid",  # Hybrid attention + LSTM
    n_quantiles=3,  # [10th, 50th, 90th percentiles] for uncertainty
    attention_stack=[
        ("cross_attention", {"heads": 4, "dim": 64}),
        ("self_attention", {"heads": 4, "dim": 64}),
    ],
)

# Compile
air_quality_model.compile(optimizer="adam", loss="mse", metrics=["mae", "mape"])

# Train (in real scenario, use actual data)
print("\nTraining air quality model...")
history = air_quality_model.fit(
    [static_features, dynamic_past, known_future],
    target,
    epochs=3,
    batch_size=16,
    validation_split=0.2,
    verbose=0,
)

print("Training complete!")
print(f"Final MSE: {history.history['loss'][-1]:.4f}")

In [ ]:
# Make predictions
predictions = air_quality_model.predict(
    [static_features[:5], dynamic_past[:5], known_future[:5]]
)

# Extract quantiles (if using quantile outputs)
print("\nAir Quality Predictions (next 24 hours):")
print(f"Prediction shape: {predictions.shape}")
print("\nExample prediction for location 1:")
print(f"  Mean PM2.5 (12-hour avg): {np.mean(predictions[0, 6:18, 0]):.1f} µg/m³")
print(f"  Max PM2.5 in forecast: {np.max(predictions[0, :, 0]):.1f} µg/m³")
print(f"  Min PM2.5 in forecast: {np.min(predictions[0, :, 0]):.1f} µg/m³")

## 2. Energy Demand Forecasting

### Use Case
Predict electricity demand for grid management:
- **Static features**: Building type, area, insulation level, number of units
- **Dynamic past features**: Historical load, temperature, solar irradiance, time-based features
- **Known future features**: Weather forecast, calendar information

### Application
- Power grid optimization
- Renewable energy integration
- Peak load management
- Cost minimization

In [ ]:
# Create synthetic energy demand data
batch_size = 32

# Static features: [building_type, area, insulation_level, units]
static_energy = np.random.randn(batch_size, 4)
static_energy[:, 1] = np.abs(static_energy[:, 1]) * 10000  # Area: 0-20000 m²
static_energy[:, 3] = np.abs(static_energy[:, 3]) * 100  # Units: 0-200

# Dynamic past features: [load, temp, solar, hour_sin, hour_cos]
lookback_window = 336  # 2 weeks of hourly data
n_dynamic_features = 5
dynamic_energy = np.random.randn(batch_size, lookback_window, n_dynamic_features)
dynamic_energy[:, :, 0] = np.abs(dynamic_energy[:, :, 0]) * 500  # Load: 0-1000 kW
dynamic_energy[:, :, 1] = 15 + dynamic_energy[:, :, 1] * 10  # Temp: 5-25°C
dynamic_energy[:, :, 2] = np.abs(dynamic_energy[:, :, 2]) * 800  # Solar: 0-1000 W/m²
# Hour encoding (sine/cosine for 24-hour cycle)
hours = np.arange(lookback_window) % 24
dynamic_energy[:, :, 3] = np.sin(2 * np.pi * hours / 24)  # Hour sine
dynamic_energy[:, :, 4] = np.cos(2 * np.pi * hours / 24)  # Hour cosine

# Known future features: [forecasted_temp, day_type]
forecast_horizon = 48  # 48-hour forecast
known_future_energy = np.random.randn(batch_size, forecast_horizon, 2)
known_future_energy[:, :, 0] = 15 + known_future_energy[:, :, 0] * 10  # Temp
known_future_energy[:, :, 1] = (np.arange(forecast_horizon) % 7 < 5).astype(
    float
)  # Weekday

# Target: Energy demand for next 48 hours
target_energy = (
    np.abs(np.random.randn(batch_size, forecast_horizon, 1)) * 500 + 200
)  # 200-700 kW

print("Energy Demand Data Shapes:")
print(f"  Static features: {static_energy.shape}")
print(f"  Dynamic past: {dynamic_energy.shape}")
print(f"  Known future: {known_future_energy.shape}")
print(f"  Target: {target_energy.shape}")

In [ ]:
# Configure BaseAttentive for energy forecasting
energy_model = BaseAttentive(
    static_dim=4,
    dynamic_dim=5,
    future_dim=2,
    forecast_horizon=48,
    lookback_window=336,
    mode="hybrid",
    n_quantiles=5,  # More quantiles for uncertainty quantification
    attention_stack=[
        ("cross_attention", {"heads": 8, "dim": 128}),
        ("self_attention", {"heads": 8, "dim": 128}),
    ],
)

energy_model.compile(optimizer="adam", loss="mse", metrics=["mae"])

print("Training energy demand model...")
history_energy = energy_model.fit(
    [static_energy, dynamic_energy, known_future_energy],
    target_energy,
    epochs=3,
    batch_size=16,
    validation_split=0.2,
    verbose=0,
)

print("Training complete!")

In [ ]:
# Make predictions
energy_predictions = energy_model.predict(
    [static_energy[:3], dynamic_energy[:3], known_future_energy[:3]]
)

print("\nEnergy Demand Predictions (next 48 hours):")
print(f"Prediction shape: {energy_predictions.shape}")
print("\nExample forecast for building 1:")
print(f"  Peak demand: {np.max(energy_predictions[0, :, 0]):.0f} kW")
print(f"  Min demand: {np.min(energy_predictions[0, :, 0]):.0f} kW")
print(f"  Average demand: {np.mean(energy_predictions[0, :, 0]):.0f} kW")
print(f"  Peak hour: {np.argmax(energy_predictions[0, :, 0])}:00")

## 3. Weather Prediction

### Use Case
Multi-step weather forecasting:
- **Static features**: Geographic location (lat, lon), elevation, terrain type
- **Dynamic past features**: Historical weather (temp, pressure, humidity, wind)
- **Known future features**: Seasonal data, calendar features

### Application
- Weather forecasting services
- Climate monitoring
- Extreme weather alerts
- Agricultural planning

In [ ]:
# Create synthetic weather data
batch_size = 32

# Static features: [latitude, longitude, elevation, terrain_type]
static_weather = np.random.randn(batch_size, 4)
static_weather[:, 0] = 40 + static_weather[:, 0] * 10  # Latitude: 30-50°N
static_weather[:, 1] = -100 + static_weather[:, 1] * 30  # Longitude: -130 to -70°
static_weather[:, 2] = np.abs(static_weather[:, 2]) * 2000  # Elevation: 0-4000m
static_weather[:, 3] = (static_weather[:, 3] > 0).astype(float)  # Terrain: 0-1

# Dynamic past features: [temp, pressure, humidity, wind_speed, wind_dir]
lookback_window = 120  # 5 days of 2-hourly data
dynamic_weather = np.random.randn(batch_size, lookback_window, 5)
dynamic_weather[:, :, 0] = 15 + dynamic_weather[:, :, 0] * 5  # Temp: 10-20°C
dynamic_weather[:, :, 1] = (
    1013 + dynamic_weather[:, :, 1] * 10
)  # Pressure: 1003-1023 hPa
dynamic_weather[:, :, 2] = (
    60 + np.abs(dynamic_weather[:, :, 2]) * 20
)  # Humidity: 20-100%
dynamic_weather[:, :, 3] = np.abs(dynamic_weather[:, :, 3]) * 5  # Wind: 0-15 m/s
dynamic_weather[:, :, 4] = np.abs(dynamic_weather[:, :, 4]) * 180  # Direction: 0-360°

# Known future features: [season, month]
forecast_horizon = 30  # 30-hour forecast (2-hourly)
known_future_weather = np.random.randn(batch_size, forecast_horizon, 2)
known_future_weather[:, :, 0] = np.tile([0, 0, 1], forecast_horizon)[
    :forecast_horizon
]  # Season coding
known_future_weather[:, :, 1] = 3  # Month: March

# Target: Temperature and pressure for next 30 periods
target_weather = np.random.randn(batch_size, forecast_horizon, 2)
target_weather[:, :, 0] = 15 + target_weather[:, :, 0] * 5  # Temp
target_weather[:, :, 1] = 1013 + target_weather[:, :, 1] * 10  # Pressure

print("Weather Data Shapes:")
print(f"  Static: {static_weather.shape}")
print(f"  Dynamic past: {dynamic_weather.shape}")
print(f"  Known future: {known_future_weather.shape}")
print(f"  Target: {target_weather.shape}")

In [ ]:
# Configure and train weather model
weather_model = BaseAttentive(
    static_dim=4,
    dynamic_dim=5,
    future_dim=2,
    forecast_horizon=30,
    lookback_window=120,
    mode="transformer",  # Pure attention for weather
    output_dim=2,  # Predict temp and pressure
    attention_stack=[
        ("cross_attention", {"heads": 8, "dim": 128}),
        ("self_attention", {"heads": 8, "dim": 128}),
        ("memory_attention", {"heads": 4, "dim": 64, "memory_size": 32}),
    ],
)

weather_model.compile(optimizer="adam", loss="mse")

print("Training weather forecasting model...")
history_weather = weather_model.fit(
    [static_weather, dynamic_weather, known_future_weather],
    target_weather,
    epochs=2,
    batch_size=16,
    verbose=0,
)

print("Training complete!")
print(f"Final loss: {history_weather.history['loss'][-1]:.4f}")

## 4. Traffic Flow Prediction

### Use Case
Predict vehicle flow and congestion:
- **Static features**: Road type, speed limit, number of lanes, urban/highway
- **Dynamic past features**: Historical traffic volume, speed, congestion level
- **Known future features**: Time of day, day type, events

### Application
- Traffic management systems
- Route optimization
- Congestion prediction and alerts
- Infrastructure planning

In [ ]:
# Create synthetic traffic data
batch_size = 32

# Static features: [road_type, speed_limit, n_lanes, is_urban]
static_traffic = np.random.randn(batch_size, 4)
static_traffic[:, 0] = (np.random.rand(batch_size) > 0.5).astype(
    float
)  # Road type: 0/1
static_traffic[:, 1] = 50 + np.abs(static_traffic[:, 1]) * 50  # Speed: 50-100 km/h
static_traffic[:, 2] = np.abs(static_traffic[:, 2]) * 4 + 1  # Lanes: 1-5
static_traffic[:, 3] = (static_traffic[:, 3] > 0).astype(float)  # Urban: 0/1

# Dynamic past features: [volume, speed, congestion, incident]
lookback_window = 288  # 24 hours of 5-minute data
dynamic_traffic = np.random.randn(batch_size, lookback_window, 4)
dynamic_traffic[:, :, 0] = (
    np.abs(dynamic_traffic[:, :, 0]) * 2000
)  # Volume: 0-4000 vehicles/h
dynamic_traffic[:, :, 1] = 60 + dynamic_traffic[:, :, 1] * 20  # Speed: 20-100 km/h
dynamic_traffic[:, :, 2] = np.abs(dynamic_traffic[:, :, 2]) * 0.5  # Congestion: 0-1
dynamic_traffic[:, :, 3] = (np.random.rand(batch_size, lookback_window) > 0.95).astype(
    float
)  # Incident

# Known future features: [hour, is_weekend]
forecast_horizon = 48  # 4 hours of 5-minute forecasts
known_future_traffic = np.random.randn(batch_size, forecast_horizon, 2)
hours = np.arange(forecast_horizon) % 288 / 12
known_future_traffic[:, :, 0] = hours  # Hour of day
known_future_traffic[:, :, 1] = (
    np.arange(forecast_horizon) % 2016 // 288
) >= 5  # Weekend

# Target: Volume and speed for next 4 hours
target_traffic = np.random.randn(batch_size, forecast_horizon, 2)
target_traffic[:, :, 0] = np.abs(target_traffic[:, :, 0]) * 2000 + 500  # Volume
target_traffic[:, :, 1] = 60 + target_traffic[:, :, 1] * 20  # Speed

print("Traffic Data Shapes:")
print(f"  Static: {static_traffic.shape}")
print(f"  Dynamic past: {dynamic_traffic.shape}")
print(f"  Known future: {known_future_traffic.shape}")
print(f"  Target: {target_traffic.shape}")

In [ ]:
# Configure and train traffic model
traffic_model = BaseAttentive(
    static_dim=4,
    dynamic_dim=4,
    future_dim=2,
    forecast_horizon=48,
    lookback_window=288,
    mode="hybrid",
    output_dim=2,  # Volume and speed
    attention_stack=[
        ("cross_attention", {"heads": 4, "dim": 64}),
        ("self_attention", {"heads": 4, "dim": 64}),
    ],
)

traffic_model.compile(optimizer="adam", loss="mse", metrics=["mae"])

print("Training traffic flow model...")
history_traffic = traffic_model.fit(
    [static_traffic, dynamic_traffic, known_future_traffic],
    target_traffic,
    epochs=2,
    batch_size=16,
    verbose=0,
)

print("Training complete!")

# Make predictions
traffic_pred = traffic_model.predict(
    [static_traffic[:2], dynamic_traffic[:2], known_future_traffic[:2]]
)
print("\nTraffic Predictions (next 4 hours):")
print(
    f"Predicted volume range: {np.min(traffic_pred[:, :, 0]):.0f} - {np.max(traffic_pred[:, :, 0]):.0f} vehicles/h"
)
print(
    f"Predicted speed range: {np.min(traffic_pred[:, :, 1]):.1f} - {np.max(traffic_pred[:, :, 1]):.1f} km/h"
)

## Summary

BaseAttentive can be used as a standalone forecasting engine for diverse applications:

| Application | Static Features | Dynamic Past | Future Features | Prediction |
|---|---|---|---|---|
| Air Quality | Location, terrain | Historical pollution | Weather forecast | PM2.5, NO₂ levels |
| Energy | Building type, area | Historical load | Weather, calendar | Electricity demand |
| Weather | Geography | Historical weather | Seasons | Temperature, pressure |
| Traffic | Road properties | Historical flow | Time, events | Volume, speed |

### Key Advantages
✓ Unified architecture for different domains
✓ Attention mechanisms capture dependencies
✓ Multi-step forecasting in one shot
✓ Incorporates static context and future information
✓ GPU acceleration available
✓ Uncertainty quantification via quantile outputs